# 07 — RoBERTa Full + Ordinal Test Inference and Submission

This Kaggle-ready notebook loads the completed **RoBERTa-base full fine-tuning + hybrid ordinal loss** model from notebook 03, reproduces its validation score with the exact head/tail input builder, predicts every row of the untouched processed test set, performs label-free diagnostics, and creates the official `q2_submission.csv`.

The test set has no `overall` labels, so test Accuracy/F1 cannot be calculated. Model selection remains based on validation micro F1. Do not use prediction confidence or class distribution to claim test accuracy.

## Kaggle inputs

Attach these two inputs before running:

1. The processed dataset containing `test.csv` from notebook 02.
2. The saved output of the completed full fine-tuning run containing `roberta_full_ordinal_classifier.keras` and/or `roberta_full_ordinal_classifier_preset`.

If automatic discovery is ambiguous, set the explicit paths in the configuration cell.

In [ ]:
import os
import subprocess
import sys

os.environ['KERAS_BACKEND'] = 'tensorflow'

try:
    import keras_hub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub'])
    import keras_hub

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('KerasHub:', keras_hub.__version__)

## Configuration

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/roberta_full_ordinal_test_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 5
INFERENCE_BATCH_SIZE = 32
EXPECTED_TEST_ROWS = 20_000
MAX_LENGTH = 256
REVIEW_HEAD_WORDS = 96
REVIEW_TAIL_WORDS = 48
ORDINAL_LOSS_WEIGHT = 0.25
EXPECTED_MODEL_PARAMETERS = 124_647_173
EXPECTED_VALIDATION_MICRO_F1 = 0.6948
VALIDATION_SCORE_TOLERANCE = 0.002

# Optional explicit paths. Keep None for automatic discovery.
EXPLICIT_TEST_PATH = None
EXPLICIT_KERAS_MODEL_PATH = None
EXPLICIT_PRESET_PATH = None

keras.mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected. Enable a Kaggle GPU accelerator.')
print('GPUs:', gpus)
print('Mixed precision:', keras.mixed_precision.global_policy())

## Locate test data and saved model

Discovery validates file contents instead of trusting filenames alone. Exactly one valid candidate must exist unless an explicit path is supplied.

In [ ]:
def resolve_test_path():
    if EXPLICIT_TEST_PATH:
        return Path(EXPLICIT_TEST_PATH)
    valid = []
    for path in INPUT_ROOT.rglob('test.csv'):
        try:
            columns = pd.read_csv(path, nrows=2).columns
            if 'model_input' in columns and 'overall' not in columns:
                valid.append(path)
        except Exception:
            continue
    if len(valid) != 1:
        raise FileNotFoundError(
            f'Expected exactly one processed test.csv; found {valid}. '
            'Set EXPLICIT_TEST_PATH.'
        )
    return valid[0]

def resolve_model_sources():
    keras_path = Path(EXPLICIT_KERAS_MODEL_PATH) if EXPLICIT_KERAS_MODEL_PATH else None
    preset_path = Path(EXPLICIT_PRESET_PATH) if EXPLICIT_PRESET_PATH else None

    if keras_path is None:
        candidates = sorted(INPUT_ROOT.rglob('roberta_full_ordinal_classifier.keras'))
        if len(candidates) == 1:
            keras_path = candidates[0]
        elif len(candidates) > 1:
            raise RuntimeError(f'Multiple .keras models found: {candidates}')

    if preset_path is None:
        candidates = sorted(
            path for path in INPUT_ROOT.rglob('roberta_full_ordinal_classifier_preset')
            if path.is_dir() and (path / 'config.json').exists()
        )
        if len(candidates) == 1:
            preset_path = candidates[0]
        elif len(candidates) > 1:
            raise RuntimeError(f'Multiple saved presets found: {candidates}')

    if keras_path is None and preset_path is None:
        raise FileNotFoundError(
            'No completed RoBERTa full-ordinal .keras model or saved preset was found.'
        )
    return keras_path, preset_path

TEST_PATH = resolve_test_path()
KERAS_MODEL_PATH, PRESET_PATH = resolve_model_sources()
print('Test:', TEST_PATH)
print('Keras model:', KERAS_MODEL_PATH)
print('Preset fallback:', PRESET_PATH)

## Load and validate the processed test set

The full-fine-tuned model was trained on a head/tail text rebuilt from `reviewText`, `summary_for_model`, `verified_str`, and `vote_bucket`. This cell recreates that input exactly. `_row_id` is used solely to prove that prediction order is preserved.

In [ ]:
test_df = pd.read_csv(TEST_PATH, low_memory=False)
test_rows = len(test_df)
test_df['_row_id'] = np.arange(test_rows, dtype='int32')

assert 'overall' not in test_df.columns, 'The official test set must not contain labels.'
required_columns = {
    'reviewText', 'summary_for_model', 'verified_str', 'vote_bucket'
}
assert required_columns.issubset(test_df.columns), (required_columns, test_df.columns.tolist())
assert test_df['reviewText'].notna().all()
assert test_df['reviewText'].astype(str).str.strip().ne('').all()
assert test_df['_row_id'].eq(np.arange(test_rows)).all()
if EXPECTED_TEST_ROWS is not None:
    assert test_rows == EXPECTED_TEST_ROWS, (test_rows, EXPECTED_TEST_ROWS)

for column, missing_value in {
    'verified_str': 'unknown',
    'vote_bucket': 'missing',
    'summary_for_model': '',
}.items():
    test_df[column] = test_df[column].fillna(missing_value).astype(str)

def build_head_tail_input(row):
    review_words = str(row['reviewText']).split()
    if len(review_words) > REVIEW_HEAD_WORDS + REVIEW_TAIL_WORDS:
        review = (
            ' '.join(review_words[:REVIEW_HEAD_WORDS])
            + ' ... Review ending: '
            + ' '.join(review_words[-REVIEW_TAIL_WORDS:])
        )
    else:
        review = ' '.join(review_words)
    return (
        f"Verified: {row['verified_str']} | "
        f"Helpful votes: {row['vote_bucket']} | "
        f"Summary: {row['summary_for_model']} | "
        f"Review: {review}"
    )

test_text = test_df.apply(build_head_tail_input, axis=1).to_numpy(dtype=str)
test_df['rebuilt_model_input'] = test_text
test_ds = (
    tf.data.Dataset.from_tensor_slices(test_text)
    .batch(INFERENCE_BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
print('Test shape:', test_df.shape)
display(pd.DataFrame({'rebuilt_model_input': test_text[:3]}))

## Load the completed model

The full `.keras` artifact contains the trained backbone and classification head. The custom ordinal loss is registered so Keras can deserialize the training configuration safely; inference itself loads the model with `compile=False`. A saved KerasHub preset is retained as a fallback. The exact parameter count and sequence length are verified before validation or test inference.

In [ ]:
@keras.saving.register_keras_serializable(package='AmazonRatings')
class HybridOrdinalCrossentropy(keras.losses.Loss):
    def __init__(self, ordinal_weight=0.25, name='hybrid_ordinal_crossentropy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.ordinal_weight = float(ordinal_weight)

    def call(self, y_true, y_pred):
        y_true = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), 'int32')
        categorical = keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )
        probabilities = keras.ops.softmax(y_pred, axis=-1)
        targets = keras.ops.one_hot(y_true, NUM_CLASSES)
        predicted_cdf = keras.ops.cumsum(probabilities, axis=-1)[..., :-1]
        target_cdf = keras.ops.cumsum(targets, axis=-1)[..., :-1]
        ordinal = keras.ops.mean(
            keras.ops.square(predicted_cdf - target_cdf), axis=-1
        )
        return categorical + self.ordinal_weight * ordinal

    def get_config(self):
        config = super().get_config()
        config.update({'ordinal_weight': self.ordinal_weight})
        return config

model = None
load_errors = []

if KERAS_MODEL_PATH is not None:
    try:
        model = keras.models.load_model(
            KERAS_MODEL_PATH,
            compile=False,
            custom_objects={
                'HybridOrdinalCrossentropy': HybridOrdinalCrossentropy,
                'AmazonRatings>HybridOrdinalCrossentropy': HybridOrdinalCrossentropy,
            },
        )
        print('Loaded full .keras model.')
    except Exception as error:
        load_errors.append(f'.keras load failed: {error!r}')

if model is None and PRESET_PATH is not None:
    try:
        model = keras_hub.models.RobertaTextClassifier.from_preset(
            str(PRESET_PATH), num_classes=NUM_CLASSES, activation=None
        )
        print('Loaded saved KerasHub preset.')
    except Exception as error:
        load_errors.append(f'Preset load failed: {error!r}')

if model is None:
    raise RuntimeError('Could not load the saved model. ' + ' | '.join(load_errors))

total_parameters = int(sum(np.prod(variable.shape) for variable in model.weights))
trainable_parameters = int(sum(np.prod(variable.shape) for variable in model.trainable_weights))
print('Loaded total parameters:', f'{total_parameters:,}')
print('Loaded trainable parameters:', f'{trainable_parameters:,}')
assert total_parameters == EXPECTED_MODEL_PARAMETERS, (
    total_parameters, EXPECTED_MODEL_PARAMETERS
)

print('Model:', model.name)
print('Sequence length:', model.preprocessor.sequence_length)
assert model.preprocessor.sequence_length == MAX_LENGTH
model.summary()

## Mandatory validation reproduction

Before touching test predictions, reproduce the completed model's known validation micro F1 with the exact same head/tail builder. This catches a wrong artifact, a mismatched tokenizer, changed input construction, or a different processed dataset. Submission generation must stop if the score differs materially from `0.6948`.

In [ ]:
validation_candidates = []
for path in INPUT_ROOT.rglob('validation.csv'):
    try:
        columns = pd.read_csv(path, nrows=2).columns
        required = {
            'overall', 'reviewText', 'summary_for_model',
            'verified_str', 'vote_bucket',
        }
        if required.issubset(columns):
            validation_candidates.append(path)
    except Exception:
        continue

if len(validation_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one processed validation.csv; found {validation_candidates}'
    )

validation_path = validation_candidates[0]
validation_df = pd.read_csv(validation_path, low_memory=False)
assert len(validation_df) == 10_000
assert validation_df['overall'].between(1, 5).all()
assert validation_df['reviewText'].notna().all()
for column, missing_value in {
    'verified_str': 'unknown',
    'vote_bucket': 'missing',
    'summary_for_model': '',
}.items():
    validation_df[column] = validation_df[column].fillna(missing_value).astype(str)
validation_text = validation_df.apply(
    build_head_tail_input, axis=1
).to_numpy(dtype=str)

validation_ds = (
    tf.data.Dataset.from_tensor_slices(validation_text)
    .batch(INFERENCE_BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
validation_logits = np.asarray(model.predict(validation_ds, verbose=1), dtype='float32')
validation_predictions = np.argmax(validation_logits, axis=1) + 1
validation_micro_f1 = float(
    np.mean(validation_predictions == validation_df['overall'].to_numpy())
)
print('Reproduced validation micro F1:', validation_micro_f1)
assert abs(validation_micro_f1 - EXPECTED_VALIDATION_MICRO_F1) <= VALIDATION_SCORE_TOLERANCE, (
    validation_micro_f1, EXPECTED_VALIDATION_MICRO_F1
)
print('Validation reproduction: PASSED')

## Predict test logits and probabilities

In [ ]:
test_logits = model.predict(test_ds, verbose=1)
test_logits = np.asarray(test_logits, dtype='float32')
assert test_logits.shape == (test_rows, NUM_CLASSES), test_logits.shape
assert np.isfinite(test_logits).all()

test_probabilities = tf.nn.softmax(test_logits, axis=-1).numpy()
test_prediction_zero_based = np.argmax(test_probabilities, axis=1)
test_predictions = test_prediction_zero_based + 1
test_confidence = np.max(test_probabilities, axis=1)
test_entropy = -np.sum(
    test_probabilities * np.log(np.clip(test_probabilities, 1e-9, 1.0)), axis=1
)

assert len(test_predictions) == test_rows
assert np.isin(test_predictions, [1, 2, 3, 4, 5]).all()
print('Prediction complete.')

## Label-free test diagnostics

These checks can detect broken inference, NaNs, extreme class collapse, or unexpectedly uncertain predictions. They are not test performance metrics because true labels are unavailable.

In [ ]:
prediction_distribution = (
    pd.Series(test_predictions, name='predicted')
    .value_counts()
    .sort_index()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
    .to_frame('count')
)
prediction_distribution['percentage'] = (
    prediction_distribution['count'] / test_rows * 100
).round(2)
display(prediction_distribution)

confidence_summary = pd.Series(test_confidence).describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
display(confidence_summary.to_frame('confidence'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(
    x=prediction_distribution.index,
    y=prediction_distribution['count'],
    ax=axes[0], color='steelblue'
)
axes[0].set(title='Predicted test rating distribution', xlabel='Rating', ylabel='Rows')
sns.histplot(test_confidence, bins=50, ax=axes[1], color='darkorange')
axes[1].set(title='Maximum predicted probability', xlabel='Confidence')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_prediction_diagnostics.png', dpi=160)
plt.show()

## Save detailed predictions and official submission

The detailed CSV is for analysis only. The official submission must contain exactly one column named `predicted` and no index.

In [ ]:
details = pd.DataFrame({
    'row_id': test_df['_row_id'].to_numpy(),
    'predicted': test_predictions,
    'confidence': test_confidence,
    'entropy': test_entropy,
})
for class_index in range(NUM_CLASSES):
    details[f'prob_rating_{class_index + 1}'] = test_probabilities[:, class_index]

submission = pd.DataFrame({'predicted': test_predictions})

assert details['row_id'].eq(np.arange(test_rows)).all()
assert submission.shape == (test_rows, 1)
assert submission.columns.tolist() == ['predicted']
assert submission['predicted'].between(1, 5).all()
assert not submission['predicted'].isna().any()

submission_path = OUTPUT_DIR / 'q2_submission.csv'
details_path = OUTPUT_DIR / 'test_predictions_detailed.csv'
distribution_path = OUTPUT_DIR / 'test_prediction_distribution.csv'
submission.to_csv(submission_path, index=False)
details.to_csv(details_path, index=False)
prediction_distribution.to_csv(distribution_path, index_label='predicted')

print('Saved:', submission_path)
print('Saved:', details_path)
print('Saved:', distribution_path)
display(submission.head())

## Inspect the least-confident predictions

This is qualitative error-risk analysis only; without labels we cannot know whether these predictions are wrong.

In [ ]:
analysis_columns = [
    column for column in [
        'reviewText', 'summary', 'verified_str', 'vote_bucket',
        'summary_for_model', 'rebuilt_model_input',
    ]
    if column in test_df.columns
]
low_confidence_rows = (
    details.nsmallest(50, 'confidence')
    .merge(test_df[['_row_id'] + analysis_columns], left_on='row_id', right_on='_row_id')
    .drop(columns='_row_id')
)
low_confidence_rows.to_csv(OUTPUT_DIR / 'lowest_confidence_examples.csv', index=False)
display(low_confidence_rows.head(10))

## Output contract

The file submitted for scoring is `/kaggle/working/roberta_full_ordinal_test_inference/q2_submission.csv`. Do not submit `test_predictions_detailed.csv`; it intentionally contains additional diagnostic columns.